In [ ]:
# run this notebook after 4_make_loadings_ukb_style_R
# this notebook should be run on ukb rap
# use this notebook to project ukb samples onto the hgdp-1kg pc-space 
# make sure to upload pca_variants_acaf_DRAGEN.tsv (generated in 4_make_loadings_ukb_style_R) to the ukb rap
# make sure to upload hgdp_1000g_variant_loadings_acaf_af_added.tsv (generated in 3_project_aou_python) to the ukb rap
# after this, run 6_analyze_pca_R

In [ ]:
# below is the script, generate_ukb_pca_vcf.sh,
# to be submitted as a job on ukb rap to generate a vcf to load into hail

#!/bin/bash

# for chr in {1..22}; do
#   plink2 \
#     --pfile /mnt/project/Lin/pca/chr${i}_hq_snps_aou_hgdp_1000g_ukbb_DRAGEN \
#     --extract /mnt/project/Lin/pca/pca_BIALLELIC/pgen_pca_variants/loadings/pca_variants_acaf_DRAGEN.tsv \
#     --make-pgen \
#     --out chr${chr}_filtered
# done

# # create a list of filtered pgen files for merging
# for chr in {1..22}; do
#   echo "chr${chr}_filtered" >> pgen_list.txt
# done

# # merge all filtered chromosomes
# plink2 \
#   --pmerge-list pgen_list.txt \
#   --make-pgen \
#   --out merged_all_chr

# # convert merged pgen to vcf and compress
# plink2 \
#   --pfile merged_all_chr \
#   --export vcf bgz \
#   --out ukb_pca_biallelic

# # clean up intermediate files (optional)
# rm chr*_filtered.pgen chr*_filtered.pvar chr*_filtered.psam
# rm merged_all_chr.pgen merged_all_chr.pvar merged_all_chr.psam
# rm pgen_list.txt

# echo "Done! Output: ukb_pca_biallelic.vcf.gz"

In [ ]:
from pyspark.sql import SparkSession
import hail as hl
import os
import pandas as pd 
import dxpy

builder = (
    SparkSession
    .builder
    .enableHiveSupport()
)
spark = builder.getOrCreate()
hl.init(sc=spark.sparkContext)

In [ ]:
hl.default_reference('GRCh38')

In [ ]:
# convert vcf to hail mt 

vcf_path = f'file:///mnt/project/Lin/pca/pca_BIALLELIC/pgen_pca_variants/after_ld_prune/ukb_pca_biallelic.vcf.gz'
# load as matrixtable
recode = {str(i): f"chr{i}" for i in range(1, 23)}
mt = hl.import_vcf(vcf_path, force_bgz=True, reference_genome='GRCh38', contig_recoding=recode)
mt = mt.repartition(500)
db_name = "ukb_aou_gxe"
mt_name = "pca_pruned_all_ukbb_biallelic.mt"
stmt = f"CREATE DATABASE IF NOT EXISTS {db_name} LOCATION 'dnax://'"
print(stmt)
spark.sql(stmt).show()
# find database id of newly created database using dxpy method
db_uri = dxpy.find_one_data_object(name=f"{db_name}", classname="database")['id']
url = f"dnax://{db_uri}/{mt_name}" # note: the dnax url must follow this format to properly save mt to dnax
mt.write(url) # note: output should describe size of mt (i.e. number of rows, columns, partitions) 

In [ ]:
mt_name = f"pca_pruned_all_ukbb_biallelic.mt"
url = f"dnax://{db_uri}/{mt_name}"
mt = hl.read_matrix_table(url)

In [ ]:
# correct loadings to match ukb ref/alt config

loadings_ht = hl.import_table("file:///mnt/project/Lin/pca/pca_BIALLELIC/pca_files/hgdp_1000g_variant_loadings_acaf_af_added.tsv", impute=True)
split_vid = loadings_ht.locus.split(":")
loadings_ht = loadings_ht.annotate(
    locus_new = hl.locus(split_vid[0], hl.int(split_vid[1]), reference_genome='GRCh38'),
    alleles_new = hl.parse_json(loadings_ht.alleles_new, hl.tarray(hl.tstr)),
    loadings = hl.parse_json(loadings_ht.loadings, hl.tarray(hl.tfloat64)))


loadings_ht = loadings_ht.key_by(locus=loadings_ht.locus_new)

mt_name = f"pca_pruned_all_ukbb_biallelic.mt"
url = f"dnax://{db_uri}/{mt_name}"
mt = hl.read_matrix_table(url)

# annotate mt with loadings data (from previous step)
mt = mt.annotate_rows(loadings_data=loadings_ht[mt.locus])

# create corrected loadings table
corrected_loadings = mt.rows()

corrected_loadings = corrected_loadings.select(
    locus_new=corrected_loadings.locus,
    alleles_new=corrected_loadings.alleles,  # use mt.alleles (not loadings.alleles)
    loadings=hl.if_else(
        corrected_loadings.alleles == corrected_loadings.loadings_data.alleles_new,
        corrected_loadings.loadings_data.loadings,
        corrected_loadings.loadings_data.loadings.map(lambda x: -x)
    ),
    af=hl.if_else(
        corrected_loadings.alleles == corrected_loadings.loadings_data.alleles_new,
        corrected_loadings.loadings_data.af,
        1 - corrected_loadings.loadings_data.af
    ),
    updated=hl.if_else(
        corrected_loadings.alleles == corrected_loadings.loadings_data.alleles_new,
        "NO",
        "YES"
    )
)

corrected_loadings = corrected_loadings.key_by(locus=corrected_loadings.locus_new, alleles=corrected_loadings.alleles_new)
ht_name = "loadings_biallelic_corrected.ht"
url = f"dnax://{db_uri}/{ht_name}"
corrected_loadings = corrected_loadings.checkpoint(url)

In [ ]:
# split mt to 10

sample_ids = mt.s.collect()
chunks = [sample_ids[i::10] for i in range(10)]
for i in range(10):
    print(i)
    chunk = chunks[i]
    mt_part = mt.filter_cols(hl.literal(chunk).contains(mt.s))
    db_name = "ukb_aou_gxe"
    mt_name = f"pca_pruned_all_ukbb_biallelic_{i}.mt"
    stmt = f"CREATE DATABASE IF NOT EXISTS {db_name} LOCATION 'dnax://'"
    print(stmt)
    spark.sql(stmt).show()
    # find database id of newly created database using dxpy method
    db_uri = dxpy.find_one_data_object(name=f"{db_name}", classname="database")['id']
    url = f"dnax://{db_uri}/{mt_name}" # note: the dnax url must follow this format to properly save mt to dnax
    mt_part.write(url) # note: output should describe size of mt (i.e. number of rows, columns, partitions) 

In [ ]:
# project 10 parts 

ht_name = "loadings_biallelic_corrected.ht"
url = f"dnax://{db_uri}/{ht_name}"
corrected_loadings = hl.read_table(url)
for i in range(10):
    print(i)
    mt_name = f"pca_pruned_all_ukbb_biallelic_{i}.mt"
    url = f"dnax://{db_uri}/{mt_name}"
    mt = hl.read_matrix_table(url)
    ht_in_mt = corrected_loadings.semi_join(mt.rows())
    n_matching_variants = ht_in_mt.count()
    print(f"Number of variants in ht also found in mt: {n_matching_variants}")
    ht = hl.experimental.pc_project(mt.GT, ht_in_mt.loadings, ht_in_mt.af)
    ht_name = f"ukb_pcs_{i}_biallelic.tsv"
    ht.export(ht_name) 

In [ ]:
%%bash
# copy sample files from hdfs to the jupyterlab execution environment file system

hdfs dfs -get ./ukb_pcs_*_biallelic.tsv .